# Notebook19 shared definitions
Tagged imports only. Purpose: source identity, immutable records and local namespace binding for the bounded computation/gate pilot. Inputs: frozen notebook exports. Outputs: definitions only; no training or data access during imports. Checkpoints/resume belong to the main Notebook19 workflow.

In [ ]:
from pathlib import Path
import copy, hashlib, json, os, sys, time, types
if "__file__" not in globals():
    __file__=str(Path(os.environ.get("FMM_JEPA_ROOT","/home/benjamin/workspace/ahr_go/fmm_jeppa"))/"experiments/19_compute_and_gate/common.ipynb")
ROOT=Path(__file__).resolve().parents[2]
_CACHE={}

def export_hash(path):
    nb=json.loads(Path(path).read_text())
    source="\n".join("".join(c["source"])for c in nb["cells"]if c["cell_type"]=="code"and"ahr-export"in c.get("metadata",{}).get("tags",[]))
    return hashlib.sha256(source.encode()).hexdigest()

def load(root,relative,name):
    path=(Path(root)/relative).resolve();key=(str(path),name)
    if key not in _CACHE:
        module=types.ModuleType(name);module.__file__=str(path)
        for cell in json.loads(path.read_text())["cells"]:
            if cell["cell_type"]=="code"and"ahr-export"in cell.get("metadata",{}).get("tags",[]):
                exec(compile("".join(cell["source"]),str(path)+"#ahr-export","exec"),module.__dict__)
        _CACHE[key]=module
    return _CACHE[key]

def bootstrap(root=ROOT):
    module=load(root,"fmm_jepa/bootstrap.ipynb","fmm19_bootstrap")
    module.install_notebook_package(Path(root));return module

def pipeline(root=ROOT):
    bootstrap(root);return load(root,"experiments/17_fixed_kernel/pipeline.ipynb","fmm19_pipeline")

def runner(root=ROOT):
    bootstrap(root);return load(root,"experiments/19_compute_and_gate/runner.ipynb","fmm19_runner")

def modules(root=ROOT):return runner(root).modules(root)
def source_hashes(root=ROOT):return runner(root).source_hashes(root)
def canonical(value):return json.loads(json.dumps(value,sort_keys=True,allow_nan=False))
def digest(value):return hashlib.sha256(json.dumps(canonical(value),sort_keys=True,separators=(",",":")).encode()).hexdigest()
def read(path):return json.loads(Path(path).read_text())
def file_record(path):
    path=Path(path).resolve();h=hashlib.sha256()
    with path.open("rb")as f:
        for block in iter(lambda:f.read(2**20),b""):h.update(block)
    return {"path":str(path),"sha256":h.hexdigest(),"bytes":path.stat().st_size}

def atomic_json(path,value):
    bootstrap(ROOT)
    from fmm_jepa.training import atomic_json as write
    return write(Path(path),canonical(value))

def immutable(path,value):
    path=Path(path);value=canonical(value)
    if path.exists():
        if read(path)!=value:raise ValueError("Immutable experiment record changed: "+str(path))
    else:atomic_json(path,value)
    return value

def check_deadline(deadline,required=0.):
    if deadline is not None and time.time()+required>=float(deadline):raise TimeoutError("Original phase deadline exhausted")

def environment(config):
    R,_,_=modules(ROOT);return R.configure_environment(config)

def data_bundle(root,artifact,config,verify_images=True):return runner(root).data_bundle(root,artifact,config,verify_images=verify_images)
def builder(root,selected):return runner(root).builder(root,selected)
def bind_backend(root,selected):return runner(root).bind(root,selected)

def handoff(root=ROOT,source_run="vit-geometry-001"):
    path=Path(root)/"artifacts/notebook_15"/source_run/"handoff_15_to_16.json"
    if not path.exists():raise FileNotFoundError("Completed Notebook15 frozen reference required")
    value=read(path)
    if value.get("status")!="COMPLETE"or value.get("fixture_source"):raise ValueError("Real completed reference required; quality gate is not reused for this new kernel")
    return {**value,"handoff_record":file_record(path)}

def learned_source(root=ROOT,handoff_value=None,variant="vit"):
    import torch
    value=handoff_value or handoff(root)
    plan_path=value["candidate_plan"];training_path=value["training_result"]
    if file_record(plan_path)["sha256"]!=value["plan_sha256"]or file_record(training_path)["sha256"]!=value["training_sha256"]:raise ValueError("Frozen15 source documents changed")
    plan=read(plan_path);training=read(training_path);seed=min(plan["seeds"])
    arm=next(a for a in plan["arms"]if int(a["seed"])==int(seed)and a["variant"]==variant)
    record=training["arms"][arm["id"]]["milestones"][str(plan["updates"])]
    if file_record(record["path"])["sha256"]!=record["sha256"]:raise ValueError("Frozen15 checkpoint changed")
    R,_,_=modules(root);config=copy.deepcopy(arm["config"])
    model=R._default_build_model(config,arm["kind"]).cpu()
    payload=torch.load(record["path"],map_location="cpu",weights_only=False);model.load_state_dict(payload["model"])
    return model,config,{"role":"FROZEN_NOTEBOOK15_TRAIN_ONLY_FIRST_REGISTERED_VIT_SEED","seed":seed,"variant":variant,"checkpoint":record,"plan":file_record(plan_path),"training":file_record(training_path),"official_test":"NOT_READ"}
